# Data Behavior Refactor Audit

This notebook compares datamodule/sample behavior between two separate LFM checkouts, for example:

- base repo: `ibm_model`
- refactor repo: `ibm_oop_refactor`

It intentionally runs each repo audit in a separate Python subprocess so imports from one checkout cannot pollute the other checkout through `sys.modules`.

The audit does not train. It instantiates configured datamodules, samples a few items from train/val/test, summarizes tensors/paths/targets, and compares JSON summaries.

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import subprocess
import sys
import tempfile
from pprint import pprint

# Edit these paths before running.
BASE_REPO_ROOT = Path(r"C:/path/to/base/lfm")
OOP_REPO_ROOT = Path(r"C:/path/to/oop_refactor/lfm")
DATA_ROOT = Path(r"C:/path/to/data_root")

# Usually both repos should read the same data root. Override these only if the old branch
# needs a different prepared data layout.
BASE_DATA_ROOT = DATA_ROOT
OOP_DATA_ROOT = DATA_ROOT

# Use the same Python executable for both repos unless you need different envs.
BASE_PYTHON = sys.executable
OOP_PYTHON = sys.executable

OUTPUT_DIR = Path.cwd() / "lfm_data_behavior_audit_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Keep this small. This is intended to catch preprocessing/target contract drift, not benchmark training.
N_SAMPLES_PER_SPLIT = 3
TARGET_SIZE = 256

# Common defaults for current pre-split data roots:
#   data_root/{train,val,test}/chips
#   data_root/{train,val,test}/labels
COMMON_FILE_KWARGS = {
    "image_glob": "*.tif",
    "label_glob": "*_label.*",
    "image_suffix": "_input_wac_static_chip",
    "label_suffix": "_label",
}

print("base repo:", BASE_REPO_ROOT)
print("oop repo:", OOP_REPO_ROOT)
print("base data root:", BASE_DATA_ROOT)
print("oop data root:", OOP_DATA_ROOT)
print("output dir:", OUTPUT_DIR)

## Audit Cases

Each case names a datamodule class and constructor kwargs. If the base branch has older module paths, edit the `base_module` / `base_class` fields for that case without changing the refactor side.

If one side needs different constructor kwargs, add `base_kwargs` or `oop_kwargs` to that case. Those override `kwargs` for only that side.

Disable cases that do not apply to your dataset by setting `enabled: False`.

In [ ]:
AUDIT_CASES = [
    {
        "name": "toy_semantic",
        "enabled": True,
        "base_module": "lfm.toy_model.sem_seg.lightning_wrappers.toy_sem_seg_datamodule",
        "base_class": "LunarSemanticSegmentationSplitDataModule",
        "oop_module": "lfm.toy_model.sem_seg.lightning_wrappers.toy_sem_seg_datamodule",
        "oop_class": "LunarSemanticSegmentationSplitDataModule",
        "kwargs": {
            "batch_size": 2,
            "num_workers": 0,
            "target_size": [TARGET_SIZE, TARGET_SIZE],
            "spatial_transform": "crop",
            "image_file_type": ".tif",
            "label_file_type": ".npy",
            "label_npz_key": "mask",
            "binarize_label": False,
            "normalize_inputs": False,
            "scale_inputs": True,
            "max_train_samples": 8,
            "max_val_samples": 8,
            "max_test_samples": 8,
            **COMMON_FILE_KWARGS,
        },
    },
    {
        "name": "toy_instance_mask2former",
        "enabled": True,
        "base_module": "lfm.toy_model.inst_seg.lightning_wrappers.toy_instance_seg_datamodule",
        "base_class": "ToyInstanceSegSplitDataModule",
        "oop_module": "lfm.toy_model.inst_seg.lightning_wrappers.toy_instance_seg_datamodule",
        "oop_class": "ToyInstanceSegSplitDataModule",
        "kwargs": {
            "batch_size": 2,
            "num_workers": 0,
            "target_size": TARGET_SIZE,
            "normalize_inputs": False,
            "scale_inputs": True,
            "mask_shift": [0, 0],
            "max_train_samples": 8,
            "max_val_samples": 8,
            "max_test_samples": 8,
            **COMMON_FILE_KWARGS,
        },
    },
    {
        "name": "toy_instance_mask_rcnn",
        "enabled": True,
        "base_module": "lfm.toy_model.inst_seg.lightning_wrappers.toy_dino_mask_rcnn_datamodule",
        "base_class": "ToyDinoMaskRCNNSplitDataModule",
        "oop_module": "lfm.toy_model.inst_seg.lightning_wrappers.toy_dino_mask_rcnn_datamodule",
        "oop_class": "ToyDinoMaskRCNNSplitDataModule",
        "kwargs": {
            "batch_size": 2,
            "num_workers": 0,
            "target_size": TARGET_SIZE,
            "normalize_inputs": False,
            "scale_inputs": True,
            "mask_shift": [0, 0],
            "max_train_samples": 8,
            "max_val_samples": 8,
            "max_test_samples": 8,
            **COMMON_FILE_KWARGS,
        },
    },
    {
        "name": "graha_semantic",
        "enabled": True,
        "base_module": "lfm.full_model.sem_seg.semantic_mask_datamodule",
        "base_class": "LunarSemanticMaskSegmentationDatamodule",
        "oop_module": "lfm.full_model.sem_seg.semantic_mask_datamodule",
        "oop_class": "LunarSemanticMaskSegmentationDatamodule",
        "kwargs": {
            "batch_size": 2,
            "num_workers": 0,
            "crop_size": TARGET_SIZE,
            "means": None,
            "stds": None,
            "binarize_mask": True,
            "max_train_samples": 8,
            "max_val_samples": 8,
            "max_test_samples": 8,
            "no_data_replace": 0.0,
            "no_label_replace": None,
            **COMMON_FILE_KWARGS,
        },
    },
    {
        "name": "graha_instance_object_detection",
        "enabled": True,
        "base_module": "lfm.full_model.inst_seg.instance_mask_datamodule",
        "base_class": "LunarObjectDetectionInstanceMaskDatamodule",
        "oop_module": "lfm.full_model.inst_seg.instance_mask_datamodule",
        "oop_class": "LunarObjectDetectionInstanceMaskDatamodule",
        "kwargs": {
            "batch_size": 2,
            "num_workers": 0,
            "crop_size": TARGET_SIZE,
            "means": None,
            "stds": None,
            "target_box_format": "xyxy",
            "max_train_samples": 8,
            "max_val_samples": 8,
            "max_test_samples": 8,
            "no_data_replace": 0.0,
            "no_label_replace": None,
            "mask_shift": [0, 0],
            **COMMON_FILE_KWARGS,
        },
    },
]

enabled_cases = [case for case in AUDIT_CASES if case.get("enabled", True)]
print("enabled cases:", [case["name"] for case in enabled_cases])

In [ ]:
RUNNER_CODE = r'''
from __future__ import annotations

import argparse
import contextlib
import importlib
import json
import os
import sys
from pathlib import Path

import numpy as np
import torch


def tensor_summary(value, *, max_unique=20):
    tensor = value.detach().cpu()
    summary = {
        "kind": "tensor",
        "shape": list(tensor.shape),
        "dtype": str(tensor.dtype),
        "numel": int(tensor.numel()),
    }
    if tensor.numel() == 0:
        return summary
    numeric = tensor.float() if not tensor.is_floating_point() else tensor
    finite = torch.isfinite(numeric)
    summary["nonfinite_count"] = int((~finite).sum().item())
    if torch.any(finite):
        valid = numeric[finite]
        summary.update(
            {
                "min": round(float(valid.min().item()), 8),
                "max": round(float(valid.max().item()), 8),
                "mean": round(float(valid.mean().item()), 8),
                "std": round(float(valid.std(unbiased=False).item()), 8),
            }
        )
    if tensor.ndim <= 2 or tensor.numel() <= 4096:
        unique = torch.unique(tensor)
        if unique.numel() <= max_unique:
            summary["unique_values"] = [float(x) if tensor.is_floating_point() else int(x) for x in unique.tolist()]
        else:
            summary["unique_count"] = int(unique.numel())
            summary["unique_head"] = [float(x) if tensor.is_floating_point() else int(x) for x in unique[:max_unique].tolist()]
    return summary


def array_summary(value):
    return tensor_summary(torch.as_tensor(value))


def summarize(value, *, depth=0):
    if depth > 5:
        return {"kind": "max_depth", "repr": repr(type(value))}
    if isinstance(value, torch.Tensor):
        return tensor_summary(value)
    if isinstance(value, np.ndarray):
        out = array_summary(value)
        out["kind"] = "ndarray"
        return out
    if isinstance(value, (str, os.PathLike)):
        path = str(value)
        return {"kind": "path", "name": Path(path).name, "suffix": Path(path).suffix, "path": path}
    if isinstance(value, dict):
        return {str(key): summarize(val, depth=depth + 1) for key, val in sorted(value.items(), key=lambda kv: str(kv[0]))}
    if isinstance(value, (list, tuple)):
        return {
            "kind": type(value).__name__,
            "length": len(value),
            "items": [summarize(item, depth=depth + 1) for item in list(value)[:8]],
        }
    if isinstance(value, (int, float, bool)) or value is None:
        return value
    return {"kind": type(value).__name__, "repr": repr(value)[:200]}


def get_dataset(datamodule, split):
    return getattr(datamodule, f"{split}_dataset", None)


def get_loader(datamodule, split):
    if split == "train":
        return datamodule.train_dataloader()
    if split == "val":
        return datamodule.val_dataloader()
    if split == "test":
        return datamodule.test_dataloader()
    raise ValueError(split)


def audit_case(repo_root, data_root, case, n_samples):
    sys.path.insert(0, str(repo_root))
    module = importlib.import_module(case["module"])
    cls = getattr(module, case["class"])
    kwargs = dict(case.get("kwargs", {}))
    kwargs["data_root"] = str(data_root)
    dm = cls(**kwargs)
    result = {
        "case": case["name"],
        "module": case["module"],
        "class": case["class"],
        "datamodule_type": f"{type(dm).__module__}.{type(dm).__name__}",
        "weight_assignments": None,
        "splits": {},
    }
    try:
        dm.setup(None)
    except Exception as exc:
        result["setup_error"] = {"type": type(exc).__name__, "message": str(exc)}
        return result
    if hasattr(dm, "weight_assignments"):
        result["weight_assignments"] = getattr(dm, "weight_assignments")
    for split in ("train", "val", "test"):
        split_result = {}
        dataset = get_dataset(dm, split)
        if dataset is None:
            result["splits"][split] = {"present": False}
            continue
        split_result["present"] = True
        split_result["dataset_type"] = f"{type(dataset).__module__}.{type(dataset).__name__}"
        try:
            split_result["dataset_len"] = int(len(dataset))
        except Exception as exc:
            split_result["dataset_len_error"] = {"type": type(exc).__name__, "message": str(exc)}
            result["splits"][split] = split_result
            continue
        samples = []
        for idx in range(min(n_samples, split_result["dataset_len"])):
            try:
                samples.append(summarize(dataset[idx]))
            except Exception as exc:
                samples.append({"sample_index": idx, "error": {"type": type(exc).__name__, "message": str(exc)}})
        split_result["samples"] = samples
        try:
            batch = next(iter(get_loader(dm, split)))
            split_result["batch"] = summarize(batch)
        except Exception as exc:
            split_result["batch_error"] = {"type": type(exc).__name__, "message": str(exc)}
        result["splits"][split] = split_result
    return result


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--repo-root", required=True)
    parser.add_argument("--data-root", required=True)
    parser.add_argument("--case-json", required=True)
    parser.add_argument("--n-samples", type=int, default=3)
    args = parser.parse_args()
    repo_root = Path(args.repo_root).resolve()
    data_root = Path(args.data_root).resolve()
    case = json.loads(args.case_json)
    with contextlib.redirect_stdout(sys.stderr):
        result = audit_case(repo_root, data_root, case, args.n_samples)
    print(json.dumps(result, indent=2, sort_keys=True))


if __name__ == "__main__":
    main()
'''

RUNNER_PATH = OUTPUT_DIR / "lfm_data_audit_runner.py"
RUNNER_PATH.write_text(RUNNER_CODE, encoding="utf-8")
print("runner:", RUNNER_PATH)

In [ ]:
def run_repo_case(repo_label: str, repo_root: Path, data_root: Path, python_exe: str, case: dict) -> dict:
    side = "base" if repo_label == "base" else "oop"
    kwargs = dict(case.get("kwargs", {}))
    kwargs.update(case.get(f"{side}_kwargs", {}))
    runner_case = {
        "name": case["name"],
        "module": case[f"{side}_module"],
        "class": case[f"{side}_class"],
        "kwargs": kwargs,
    }
    cmd = [
        str(python_exe),
        str(RUNNER_PATH),
        "--repo-root",
        str(repo_root),
        "--data-root",
        str(data_root),
        "--case-json",
        json.dumps(runner_case),
        "--n-samples",
        str(N_SAMPLES_PER_SPLIT),
    ]
    completed = subprocess.run(cmd, text=True, capture_output=True)
    payload = {
        "repo_label": repo_label,
        "repo_root": str(repo_root),
        "case": case["name"],
        "returncode": completed.returncode,
        "stderr_tail": completed.stderr[-4000:],
    }
    if completed.returncode != 0:
        payload["stdout_tail"] = completed.stdout[-4000:]
        return payload
    try:
        payload["summary"] = json.loads(completed.stdout)
    except json.JSONDecodeError as exc:
        payload["json_error"] = str(exc)
        payload["stdout_tail"] = completed.stdout[-4000:]
    return payload


audit_results = {"base": {}, "oop": {}}
for case in enabled_cases:
    print(f"running {case['name']} base...")
    audit_results["base"][case["name"]] = run_repo_case("base", BASE_REPO_ROOT, BASE_DATA_ROOT, BASE_PYTHON, case)
    print(f"running {case['name']} oop...")
    audit_results["oop"][case["name"]] = run_repo_case("oop", OOP_REPO_ROOT, OOP_DATA_ROOT, OOP_PYTHON, case)

raw_path = OUTPUT_DIR / "audit_raw_results.json"
raw_path.write_text(json.dumps(audit_results, indent=2, sort_keys=True), encoding="utf-8")
print("wrote", raw_path)

In [ ]:
def flatten(obj, prefix=""):
    rows = {}
    if isinstance(obj, dict):
        for key, value in obj.items():
            rows.update(flatten(value, f"{prefix}.{key}" if prefix else str(key)))
    elif isinstance(obj, list):
        for index, value in enumerate(obj):
            rows.update(flatten(value, f"{prefix}[{index}]"))
    else:
        rows[prefix] = obj
    return rows


def values_match(a, b, *, atol=1e-6):
    if isinstance(a, float) or isinstance(b, float):
        try:
            return abs(float(a) - float(b)) <= atol
        except Exception:
            return False
    return a == b


comparison_rows = []
for case in enabled_cases:
    name = case["name"]
    base_payload = audit_results["base"][name]
    oop_payload = audit_results["oop"][name]
    if base_payload.get("returncode") != 0 or oop_payload.get("returncode") != 0:
        comparison_rows.append({
            "case": name,
            "path": "SUBPROCESS_STATUS",
            "base": base_payload.get("returncode"),
            "oop": oop_payload.get("returncode"),
            "match": False,
        })
        continue
    base_summary = base_payload.get("summary", {})
    oop_summary = oop_payload.get("summary", {})
    base_flat = flatten(base_summary)
    oop_flat = flatten(oop_summary)
    all_keys = sorted(set(base_flat) | set(oop_flat))
    for key in all_keys:
        if key in {"module", "class", "datamodule_type"}:
            continue
        base_value = base_flat.get(key, "<MISSING>")
        oop_value = oop_flat.get(key, "<MISSING>")
        match = values_match(base_value, oop_value)
        if not match:
            comparison_rows.append({
                "case": name,
                "path": key,
                "base": base_value,
                "oop": oop_value,
                "match": match,
            })

comparison_path = OUTPUT_DIR / "audit_differences.json"
comparison_path.write_text(json.dumps(comparison_rows, indent=2, sort_keys=True), encoding="utf-8")
print(f"differences: {len(comparison_rows)}")
print("wrote", comparison_path)
pprint(comparison_rows[:50])

## How To Read Results

- `audit_raw_results.json` contains the full per-repo summaries.
- `audit_differences.json` contains only mismatched flattened fields.
- Differences in `module`, `class`, and `datamodule_type` are ignored by default because those are expected to change during refactors.
- Path strings may differ if the two checkouts are in different folders; filename fields are usually more useful than full path fields.
- For Phase 2/3 behavior preservation, prioritize differences in shapes, channel counts, image min/max/mean/std, mask unique values, object counts, box shapes, and batch keys.